In [2]:
# 기본적인 상태 만들기
# 애플리케이션 예시 : 지정된 곳에서 문서를 찾고, 없다면 인터넷 검색을 하여 답변을 하는 애플리케이션
# 노드는 일단 더미로 만듦


In [8]:
# 1. 기본적인 상태

from typing import TypedDict, Annotated
from operator import add

class GraphState(TypedDict):
    query: str  # 사용자 질의
    documents: Annotated[list[str], add] # 답변을 위해 검색된 내용
    response: str    # AI 답변
    success_flag: int  # AI 답변의 품질 검토 - 1:성공, 0:실패

In [9]:
# 2. 입출력 스키마 구분

from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph

class GraphState(TypedDict):
    query: str  # 사용자 질의
    documents: Annotated[list[str], add] # 답변을 위해 검색된 내용
    response: str    # AI 답변
    success_flag: int  # AI 답변의 품질 검토 - 1:성공, 0:실패

class SearchInputState(TypedDict):
    query:str

class AnswerOutputState(TypedDict):
    response:str

def search_node(state: SearchInputState):
    def do_search(query: state["query"]):
        # do something with query
        return ["doc1", "doc2"]
    documents = do_search()
    return {
        "query" : state["query"],
        "documents" : documents
    }

def llm_invoke(state: GraphState) -> AnswerOutputState:
    def do_llm_invoke(state):
        prompt = f"user query: {state['query']}, documents: {state['documents']}"
        # do llm invoke
        return "llm response"
    response = do_llm_invoke(state)
    return {
        "response" : response
    }

# Graph 를 빌드할 때 input, output을 지정
builder = StateGraph(
    GraphState,
    input = SearchInputState,
    output = AnswerOutputState
)

In [10]:
# State 상태 업데이트 메커니즘

In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph

class GraphState(TypedDict):
    user_input: str
    intent: str
    result: str

# 사용자의 의도를 판단해, intent 채널에 업데이트
def classify_node(state: GraphState) -> dict:
    if "환불" in state["user_input"]:
        return {"intent": "refund"}
    elif "구매" in state["user_input"]:
        return {"intent": "purchase"}
    return {"intent": "general"}

# 사용자 의도에 따라 result 채널에 업데이트
def response_node(state: GraphState) -> dict:
    if state["intent"] == "refund":
        return {"result": "환불 절차를 안내합니다."}
    elif state["intent"] == "purchase":
        return {"result": "구메 절차를 안내합니다."}
    return {"result": "일반 문의로 처리합니다."}

In [32]:
# 리듀서 예시 : 삼세판 가위바위보  

# (1) count : 가위바위보 횟수  
# (2) win_logs : 각 판의 승리자를 누적한 리스트  
# (3) winner : 최종 승리자  


# 주의할 포인트
# 리턴할 때에는 dictionary 형식이어야 하고, 따라서 키는 str이어야 한다.  
# list 형태인 채널에 대해서는 list 로 리턴해야 한다.


In [33]:
from typing import Annotated
from operator import add
from typing_extensions import TypedDict
from collections import Counter
from langgraph.graph import StateGraph

class State(TypedDict):
    count: Annotated[int, add]
    win_logs: Annotated[list[str], add]
    winner: str

def first_round_node(state: State) -> State:
    return {
        "count": 1,
        "win_logs": ["철수"]
    }

def second_round_node(state: State) -> State:
    return {
        "count": 1,
        "win_logs": ["민수"]
    }

def third_round_node(state: State) -> State:
    return {
        "count": 1,
        "win_logs": ["철수"]
    }

def judge_node(state: State) -> State:
    c = Counter(state["win_logs"])
    return {
        "winner": c.most_common()[0][0]
    }



In [34]:
builder = StateGraph(State)

builder.add_node("first", first_round_node)
builder.add_node("second", second_round_node)
builder.add_node("third", third_round_node)
builder.add_node("judgement", judge_node)

builder.set_entry_point("first")
builder.add_edge("first", "second")
builder.add_edge("second", "third")
builder.add_edge("third", "judgement")

app = builder.compile()

state = State({"count":0})
app.invoke(state)

In [35]:
from typing import Annotated
from operator import add
from typing_extensions import TypedDict
from collections import Counter
from langgraph.graph import StateGraph

def judge(win_logs: list[str]) -> str:
    c = Counter(win_logs)
    return c.most_common()[0][0]

class State(TypedDict):
    count: Annotated[int, add]
    win_logs: Annotated[list[str], add]
    winner: Annotated[str, judge]

def first_round_node(state: State) -> State:
    return {
        "count": 1,
        "win_logs": ["철수"]
    }

def second_round_node(state: State) -> State:
    return {
        "count": 1,
        "win_logs": ["민수"]
    }

def third_round_node(state: State) -> State:
    return {
        "count": 1,
        "win_logs": ["철수"]
    }

def judge_node(state: State) -> State:
    c = Counter(state["win_logs"])
    return {
        "winner": state["wi
    }



{'count': 3, 'win_logs': ['철수', '민수', '철수'], 'winner': '철수'}